# Build User Histories

This notebook constructs user histories from the filtered Amazon Electronics review dataset.

For each user, interactions are ordered chronologically. A sliding window is then applied to create multiple training examples consisting of:

- User ID
- Historical interactions
- Ground-truth next interaction

These examples will later be enriched with product metadata and transformed into natural language queries.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", None)
tqdm.pandas()

In [2]:
PROJECT_DIR= Path.home()/"query-conditioned-recommender"
PROCESSED= PROJECT_DIR/"data"/"processed"
REVIEWS= PROCESSED/"filtered_reviews.parquet"
OUTPUT= PROCESSED/"user_histories.parquet"

In [3]:
reviews= pd.read_parquet(REVIEWS)
print(reviews.shape)

(7509, 7)


In [4]:
reviews.head()

,user_id,parent_asin,timestamp,rating,verified_purchase,title,text
0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B075B5T7L5,2015-09-01 12:14:16,5,True,I can assure that this micro sd card installed...,After reading so many other comments online to...
1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00WA4PSEM,2015-09-01 12:23:56,5,True,Big sound from a tiny Bluetooth speaker.,Paired up easily with my phone and has excelle...
2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B09VS4V18K,2016-10-02 21:27:51,5,True,"I've never had a WD drive die on me yet, and I...",This WD 1 TB Performance Desktop Hard Disk get...
3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00G2UTVH0,2016-10-02 22:15:43,5,True,These fit my 5 Panasonic Handsets when the bat...,These fit my 5 Panasonic Handsets and worked p...
4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B07RS68PJ7,2016-10-23 09:32:31,5,True,Not just battery backup but affordable voltage...,I needed battery backup and AVR because of ele...


## Group interactions by user

Each user's interactions are sorted chronologically.

In [5]:
reviews= reviews.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
reviews.head()

,user_id,parent_asin,timestamp,rating,verified_purchase,title,text
0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B075B5T7L5,2015-09-01 12:14:16,5,True,I can assure that this micro sd card installed...,After reading so many other comments online to...
1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00WA4PSEM,2015-09-01 12:23:56,5,True,Big sound from a tiny Bluetooth speaker.,Paired up easily with my phone and has excelle...
2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B09VS4V18K,2016-10-02 21:27:51,5,True,"I've never had a WD drive die on me yet, and I...",This WD 1 TB Performance Desktop Hard Disk get...
3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00G2UTVH0,2016-10-02 22:15:43,5,True,These fit my 5 Panasonic Handsets when the bat...,These fit my 5 Panasonic Handsets and worked p...
4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B07RS68PJ7,2016-10-23 09:32:31,5,True,Not just battery backup but affordable voltage...,I needed battery backup and AVR because of ele...


## Create sliding windows

Each user contributes multiple training examples.

Example:

History:

A B C D E

↓

Ground Truth:

F

Then

History:

B C D E F

↓

Ground Truth:

G

In [6]:
WINDOW_SIZE= 5
examples= []

In [7]:
for user, group in tqdm(reviews.groupby("user_id")):
    products= group["parent_asin"].tolist()
    timestamps= group["timestamp"].tolist()
    ratings= group["rating"].tolist()
    if len(products) <= WINDOW_SIZE:
        continue
    for i in range(len(products)-WINDOW_SIZE):
        history= products[i:i+WINDOW_SIZE]
        ground_truth= products[i+WINDOW_SIZE]
        example= {
            "sample_id": len(examples),
            "user_id": user,
            "history_asins": history,
            "ground_truth_asin": ground_truth,
            "history_length": len(history),
            "history_timestamps": timestamps[i:i+WINDOW_SIZE],
            "history_ratings": ratings[i:i+WINDOW_SIZE],
            "ground_truth_timestamp": timestamps[i+WINDOW_SIZE],
            "ground_truth_rating": ratings[i+WINDOW_SIZE],
            "query": None,
            "constraint_type": None,
            "constraint_values": None
        }
        examples.append(example)

  0%|          | 0/390 [00:00<?, ?it/s]

100%|██████████| 390/390 [00:00<00:00, 4437.09it/s]


In [8]:
dataset= pd.DataFrame(examples)
dataset.shape

(5559, 12)

In [9]:
dataset.head()

,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B075B5T7L5, B00WA4PSEM, B09VS4V18K, B00G2UTVH...",B007VKII6A,5,"[2015-09-01 12:14:16, 2015-09-01 12:23:56, 201...","[5, 5, 5, 5, 5]",2017-05-21 14:36:52.000,2,None,None,None
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00WA4PSEM, B09VS4V18K, B00G2UTVH0, B07RS68PJ...",B07454F4JH,5,"[2015-09-01 12:23:56, 2016-10-02 21:27:51, 201...","[5, 5, 5, 5, 2]",2017-08-24 17:49:29.844,5,None,None,None
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B09VS4V18K, B00G2UTVH0, B07RS68PJ7, B007VKII6...",B00SMVLPIK,5,"[2016-10-02 21:27:51, 2016-10-02 22:15:43, 201...","[5, 5, 5, 2, 5]",2017-08-24 17:57:36.638,5,None,None,None
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00G2UTVH0, B07RS68PJ7, B007VKII6A, B07454F4J...",B0179DDIQI,5,"[2016-10-02 22:15:43, 2016-10-23 09:32:31, 201...","[5, 5, 2, 5, 5]",2017-08-24 18:06:02.421,5,None,None,None
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B07RS68PJ7, B007VKII6A, B07454F4JH, B00SMVLPI...",B00X088DZI,5,"[2016-10-23 09:32:31, 2017-05-21 14:36:52, 201...","[5, 2, 5, 5, 5]",2019-05-11 17:22:30.699,5,None,None,None


In [10]:
dataset.sample(5)

,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values
2211,2211,AFKQ7HTHC5LYV6PPR7ZXD7QBIZXQ,"[B00JUNH8H6, B07B3JZ2M6, B01HVHIZFG, B01JPDTZX...",B07SNBHTKD,5,"[2014-11-12 13:25:45, 2017-11-02 16:24:26.2400...","[3, 5, 4, 5, 5]",2020-11-02 15:13:19.277,5,None,None,None
4010,4010,AGO3H7RLCHVCCYG44FAHDBYPUA4Q,"[B00ZVJAF9G, B019WHZE70, B00MWHQZX0, B00E9W1UL...",B01J1GJC5M,5,"[2016-08-13 19:00:41, 2017-01-11 21:56:41, 201...","[5, 5, 5, 5, 5]",2018-09-08 02:13:56.374,5,None,None,None
1749,1749,AFGDRVPCP742YM5MMLFIKZCGNNRQ,"[B07QL578VM, B07P7GG2GC, B07Q4F3SQH, B07SBMKC8...",B081CC65ZS,5,"[2019-05-30 03:00:32.711000, 2019-06-06 20:36:...","[5, 5, 5, 5, 1]",2019-06-24 02:01:15.519,5,None,None,None
4141,4141,AGUO64P7C4QQ3NVBLJAHCGLLUCQA,"[B075KP9XHS, B00LI9DUG8, B00EO9JNWY, B07BHXYJX...",B01N66MD3P,5,"[2015-01-10 07:32:09, 2015-01-10 07:39:21, 201...","[5, 5, 5, 5, 5]",2015-06-18 07:45:09.000,5,None,None,None
5097,5097,AHQKIZLDNQ6CWF3QH5W4RU2BWS2A,"[B06XXLVKGF, B01NAWBY12, B01N75TWSA, B0BGNG129...",B01D84LZ1W,5,"[2019-06-03 19:25:12.086000, 2019-06-03 19:34:...","[4, 5, 2, 4, 5]",2019-10-27 11:28:50.000,4,None,None,None


## Dataset statistics

In [11]:
print("Examples:", len(dataset))
print("Unique users:", dataset.user_id.nunique())
print("Average history length:", dataset.history_length.mean())

Examples: 5559
Unique users: 390
Average history length: 5.0


In [12]:
dataset.to_parquet(OUTPUT,index=False)
print("Saved:", OUTPUT)

Saved: /users/adhp511/query-conditioned-recommender/data/processed/user_histories.parquet


In [13]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 5559 entries, 0 to 5558
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   sample_id               5559 non-null   int64         
 1   user_id                 5559 non-null   str           
 2   history_asins           5559 non-null   object        
 3   ground_truth_asin       5559 non-null   str           
 4   history_length          5559 non-null   int64         
 5   history_timestamps      5559 non-null   object        
 6   history_ratings         5559 non-null   object        
 7   ground_truth_timestamp  5559 non-null   datetime64[ms]
 8   ground_truth_rating     5559 non-null   int64         
 9   query                   0 non-null      object        
 10  constraint_type         0 non-null      object        
 11  constraint_values       0 non-null      object        
dtypes: datetime64[ms](1), int64(3), object(6), str(2)
memory us